# Loadsmart — Analytics Engineer Challenge

Notebook with utility functions and exports from DuckDB.

**Prerequisites** (already in `.venv`):
```
pip install duckdb pandas paramiko
```

Run the pipeline before this notebook:
```bash
python scripts/ingest.py
cd dbt && dbt run
```

## 1. Setup

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
DUCKDB_PATH = str(PROJECT_ROOT / "data" / "loadsmart.duckdb")

print(f"DuckDB: {DUCKDB_PATH}")
print(f"File exists: {Path(DUCKDB_PATH).exists()}")

DuckDB: /Users/abner.rodrigues/cases/loadsmart_case/data/loadsmart.duckdb
Arquivo existe: True


## 2. DuckDB connection

In [2]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# List available tables
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,loadsmart,main_intermediate,int_shipments,"[loadsmart_id, lane_raw, pickup_city, pickup_s...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
1,loadsmart,main_intermediate,int_shipments_enriched,"[loadsmart_id, lane_raw, pickup_city, pickup_s...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
2,loadsmart,main_mart,dim_carrier,"[CARRIER_SK, CARRIER_NAME, CARRIER_RATING, VIP...","[VARCHAR, VARCHAR, DOUBLE, BOOLEAN, BIGINT]",False
3,loadsmart,main_mart,dim_date,"[DATE_SK, DATE_DAY, YEAR, QUARTER, MONTH, MONT...","[VARCHAR, TIMESTAMP, INTEGER, INTEGER, INTEGER...",False
4,loadsmart,main_mart,dim_location,"[LOCATION_SK, CITY, STATE]","[VARCHAR, VARCHAR, VARCHAR]",False
5,loadsmart,main_mart,dim_shipper,"[SHIPPER_SK, SHIPPER_NAME]","[VARCHAR, VARCHAR]",False
6,loadsmart,main_mart,fct_shipments,"[LOADSMART_ID, CARRIER_SK, SHIPPER_SK, PICKUP_...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
7,loadsmart,main_staging,stg_shipments,"[loadsmart_id, lane_raw, pickup_city, pickup_s...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
8,loadsmart,raw,shipments,"[loadsmart_id, lane, quote_date, book_date, so...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False


## 3. `split_lane()` function

Takes a lane string in the form `"City,ST -> City,ST"` and returns a dict with origin and destination city and state.

In [3]:
def split_lane(lane: str) -> dict:
    """
    Parse a lane string in the form 'City,ST -> City,ST'.

    Parameters
    ----------
    lane : str
        Lane string, e.g. 'Chicago,IL -> Dallas,TX'

    Returns
    -------
    dict with keys:
        pickup_city, pickup_state, delivery_city, delivery_state

    Raises
    ------
    ValueError if the format is invalid.
    """
    if not lane or " -> " not in lane:
        raise ValueError(f"Invalid lane format: {lane!r}. Expected: 'City,ST -> City,ST'")

    origin, destination = lane.split(" -> ", maxsplit=1)

    def _parse_side(side: str) -> tuple[str, str]:
        parts = side.split(",", maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Malformed lane side: {side!r}")
        return parts[0].strip(), parts[1].strip()

    pickup_city, pickup_state = _parse_side(origin)
    delivery_city, delivery_state = _parse_side(destination)

    return {
        "pickup_city": pickup_city,
        "pickup_state": pickup_state,
        "delivery_city": delivery_city,
        "delivery_state": delivery_state,
    }


# --- Manual tests ---
cases = [
    ("Chicago,IL -> Dallas,TX",      {"pickup_city": "Chicago",   "pickup_state": "IL", "delivery_city": "Dallas",  "delivery_state": "TX"}),
    ("New York,NY -> Los Angeles,CA", {"pickup_city": "New York",  "pickup_state": "NY", "delivery_city": "Los Angeles", "delivery_state": "CA"}),
    (" Miami , FL ->  Atlanta , GA ", {"pickup_city": "Miami",     "pickup_state": "FL", "delivery_city": "Atlanta", "delivery_state": "GA"}),
]

for lane_str, expected in cases:
    result = split_lane(lane_str)
    assert result == expected, f"FAIL: {lane_str!r} → {result}"
    print(f"OK  {lane_str!r}")
    print(f"    {result}")

OK  'Chicago,IL -> Dallas,TX'
    {'pickup_city': 'Chicago', 'pickup_state': 'IL', 'delivery_city': 'Dallas', 'delivery_state': 'TX'}
OK  'New York,NY -> Los Angeles,CA'
    {'pickup_city': 'New York', 'pickup_state': 'NY', 'delivery_city': 'Los Angeles', 'delivery_state': 'CA'}
OK  ' Miami , FL ->  Atlanta , GA '
    {'pickup_city': 'Miami', 'pickup_state': 'FL', 'delivery_city': 'Atlanta', 'delivery_state': 'GA'}


In [4]:
# Apply to a real sample from DuckDB
sample = con.execute("""
    SELECT LANE_RAW
    FROM main_mart.fct_shipments
    LIMIT 5
""").df()

pd.DataFrame(sample["LANE_RAW"].apply(split_lane).tolist())

,pickup_city,pickup_state,delivery_city,delivery_state
0,Vernon,CA,Tracy,CA
1,Denver,CO,Lubbock,TX
2,Crete,IL,Plant City,FL
3,Vernon,CA,Tracy,CA
4,Oxnard,CA,Tolleson,AZ


In [5]:
con.close()
print("DuckDB connection closed.")

Conexão DuckDB fechada.
